# Submission 4 — Project Development Practices (5%)

**Course:** RBB2013 Digital Twin — May 2026
**Group project — SmartClean Twin:** Digital Twin of a mobile cleaning robot (topic 2)

**Team Members:**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |
**Repository:** https://github.com/KAI-UTP/smartclean-twin

**Presentation & demo video:** [https://youtu.be/zEq7L-ivMLA](https://youtu.be/zEq7L-ivMLA)

> The video walks through the whole project: problem and purpose, architecture, live Grafana dashboard, NVIDIA Omniverse 3D twin, the five AI models, what-if simulation, live fault injection, command and control, tests, CI, scaling and persistence.


## 1. Sprint planning & execution (2 cycles)

Documented in `docs/sprint-plan.md`: features, milestones, deliverables and
sprint reviews for Sprint 1 (core pipeline) and Sprint 2 (AI, Omniverse,
robustness). Task IDs (T-xx) map to commits.


## 2. Deliverables from every team member

Sprint tasks are assigned per member in `docs/sprint-plan.md` (task IDs T-01 … T-31).
Every member delivered their artefact **committed from their own GitHub account**,
so authorship is verifiable in the version-control history (Section 7).

| Member | Sprint task | Deliverable committed | GitHub account |
|---|---|---|---|
| Chan Li Kai | T-01 … T-13, T-16 … T-26, T-31 | All service code, tests, Docker, CI, Omniverse scene, docs | KAI-UTP |
| William Wong Xiao Kang | T-14 — review MQTT topics & telemetry contract | `docs/review-william.md` | williamwxk0822 |
| Irvin Chang Hou Ceng | T-15 — review state rules & AI test cases | `docs/review-irvin.md` + 3 what-if evidence screenshots | vinutp |
| Liang Yan Ee | T-27, T-28 — verify test suites, dashboard & command flow | `docs/review-liang.md` (test verification report) | renerere |
| Nurin Emelin Binti Marhisyam | T-29 — dashboard review & sprint evidence | `docs/review-nurin.md` + `docs/evidence/` screenshots | nrnemelin |

## 3. Version control & group merge

- Single shared `main` branch on GitHub with descriptive, task-linked commits
- **Group merge at end of sprint:** each member pushed/merged their own
  deliverable into `main`; conflicts resolved by rebase (`git pull --rebase`)
- Every push is gated by the CI pipeline (Section 5), so `main` is always green


## 4. Tests for each module (unit + interface / integration)

| Module | Unit tests | Integration / interface tests |
|---|---|---|
| `shared/smartclean_common` (schemas, topics) | `tests/unit/test_telemetry_schema.py` | exercised by every integration test |
| `robot-simulator` (physics, grid, commands) | `tests/unit/test_grid_map.py`, `test_simulator_commands.py` | `tests/system/test_full_flow.py` (command → ACK) |
| `state-engine` (rules, twin state) | `tests/unit/test_state_rules.py` | `tests/system/test_full_flow.py` (telemetry → state) |
| `ai-service` (models, predictor) | `tests/unit/test_ai_predictor.py` | `tests/system/test_full_flow.py` (prediction publish) |
| `command-api` (validation, MQTT publish) | `tests/unit/test_command_validation.py` | `tests/integration/test_command_api.py` |
| `telemetry-ingestion` (validate → InfluxDB) | covered via schema unit tests | `tests/integration/test_telemetry_ingestion.py` |
| Whole system | — | `tests/system/` (11 tests) + `tests/regression/` (golden snapshots) |


## 5. CI/CD — automatic build & regression on every push

`.github/workflows/ci.yml`: ruff lint → black format check → full pytest
suite → Docker image builds. A red run blocks the merge; history of green
runs on the GitHub Actions page.


## 6. Live evidence — full test suite (unit, integration, system, regression)

In [1]:
import os, subprocess, sys

env = dict(os.environ, INTEGRATION_TEST="1")
for label, target in [
    ("Unit", "tests/unit"),
    ("Integration", "tests/integration"),
    ("System", "tests/system"),
    ("Regression", "tests/regression"),
]:
    r = subprocess.run([sys.executable, "-m", "pytest", target, "-q", "--no-header"],
                       capture_output=True, text=True, cwd=".", env=env)
    tail = [l for l in r.stdout.splitlines() if l.strip()][-1:]
    print(f"{label:12s} {tail[0] if tail else 'no output':60s} exit={r.returncode}")


Unit         ============================= 81 passed in 0.98s ============================== exit=0


Integration  ============================= 11 passed in 2.07s ============================== exit=0


System       ============================= 14 passed in 22.55s ============================= exit=0


Regression   ============================= 10 passed in 0.80s ============================== exit=0


## 7. Demonstrated FAIL case (then recovery)

A test suite is only credible if it fails when the system is broken. Here the
MQTT broker is deliberately stopped, the integration tests are re-run (they
must FAIL), the broker is restarted, and the same tests are re-run (they must
PASS again). This is the pass-and-fail evidence required by the rubric.


In [2]:
import os, subprocess, sys, time

env = dict(os.environ, INTEGRATION_TEST="1")

def run_integration(label):
    r = subprocess.run([sys.executable, "-m", "pytest",
                        "tests/system/test_full_flow.py", "-q", "--no-header"],
                       capture_output=True, text=True, cwd=".", env=env)
    tail = [l for l in r.stdout.splitlines() if l.strip()][-1:]
    print(f"{label}: {tail[0] if tail else r.stdout[-200:]}  (exit {r.returncode})")
    return r.returncode

print("STEP 1 — baseline, stack healthy")
run_integration("  PASS expected ")

print("\nSTEP 2 — break the system: stop the MQTT broker")
subprocess.run(["docker", "stop", "smartclean-mosquitto"], capture_output=True, cwd=".")
time.sleep(5)
rc_broken = run_integration("  FAIL expected ")

print("\nSTEP 3 — repair: restart the broker")
subprocess.run(["docker", "start", "smartclean-mosquitto"], capture_output=True, cwd=".")
time.sleep(20)
rc_fixed = run_integration("  PASS expected ")

print()
print("Fail case demonstrated:", rc_broken != 0)
print("Recovery demonstrated: ", rc_fixed == 0)


STEP 1 — baseline, stack healthy


  PASS expected : ============================= 11 passed in 8.96s ==============================  (exit 0)

STEP 2 — break the system: stop the MQTT broker


  FAIL expected : ======================== 3 failed, 8 passed in 29.01s =========================  (exit 1)

STEP 3 — repair: restart the broker


  PASS expected : ============================= 11 passed in 9.02s ==============================  (exit 0)

Fail case demonstrated: True
Recovery demonstrated:  True


## 8. Live evidence — version-control history (all members)

In [3]:
import subprocess

r = subprocess.run(["git", "log", "--format=%h|%an|%ad|%s", "--date=short", "-16"],
                   capture_output=True, text=True, cwd=".")
for line in r.stdout.splitlines():
    parts = line.split("|")
    if len(parts) == 4:
        h, an, ad, s = parts
        print(f"{h}  {an[:26]:28s} {ad}  {s[:52]}")

print()
print("Distinct commit authors (group merge evidence):")
a = subprocess.run(["git", "log", "--format=%an"], capture_output=True, text=True, cwd=".")
for name in sorted(set(a.stdout.split(chr(10))) - {""}):
    print("  -", name)


18e8b77  KAI-UTP                      2026-07-27  docs: refresh walkthrough notebook outputs
f92f34d  KAI-UTP                      2026-07-27  docs: remove rubric-mapping tables from submission n
8612c0a  KAI-UTP                      2026-07-27  docs: add presentation video link to all notebooks
df40f3f  renerere                     2026-07-27  Add files via upload
29e829d  KAI-UTP                      2026-07-27  docs: presentation/video script with screen cues
42af2de  KAI-UTP                      2026-07-27  docs: Skilled(5) rubric-mapping table appended to ea
69a4ba9  KAI-UTP                      2026-07-27  fix: persistence test fixed window, scalable ingesti
49aba91  KAI-UTP                      2026-07-27  fix(omniverse): cast values to string before group+p
2c4a880  KAI-UTP                      2026-07-27  fix(omniverse): group() before pivot so all queried 
4ef0a54  KAI-UTP                      2026-07-26  docs: move William review into docs/ with the other 
d1ab3a2  KAI-UTP  